# Week 3 Day 1 with the newer AWS sign-up

This notebook accompanies the [course lab](../../labs/week3day1.ipynb) for students who already have a project from the newer AWS sign-up. For the Render route, use the [free version of the lab](../../labs/week3day1free.ipynb).

In my Day 1 run, the AWS CLI and boto3 authenticated as an IAM user. The plan check returned **PAID (ACTIVE)**, and AgentCore returned an empty runtime list in **Asia Pacific (Sydney), ap-southeast-2**.

## Check your project and CLI profile

Open AWS Settings. Choose **Project**, then **Overview** → **Additional Info**. Note the Region and account ID for your project. AWS assigns a home Region when it provisions the project. [AWS project regions](https://docs.aws.amazon.com/accounts/latest/reference/project-regions.html)

In a terminal, run `aws --version` and `aws sts get-caller-identity`. If you use a named AWS profile, add `--profile your-profile` to the second command. Compare the returned account ID with the one in AWS Settings. I used an IAM user configured for programmatic access. Keep credentials in your local CLI configuration. [AWS sign-up comparison](https://docs.aws.amazon.com/accounts/latest/reference/sign-up-for-aws.html) · [AWS CLI profiles](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-files.html)

Open this notebook with the course's `.venv` kernel. Set `profile_name` if you used a named profile. Replace `ap-southeast-2` with your home Region if it differs.

In [ ]:
import boto3

profile_name = None
project_region = "ap-southeast-2"
session = boto3.Session(profile_name=profile_name, region_name=project_region)

identity = session.client("sts").get_caller_identity()
principal = identity["Arn"].split(":", 5)[-1]
if principal == "root":
    raise RuntimeError("Choose a non-root CLI profile for the lab.")

print(f"AWS identity type: {principal.split('/', 1)[0]}")
print(f"AWS Region: {session.region_name}")

## Check sign-in security and costs

The newer AWS sign-up uses AWS Builder ID for console sign-in. Check MFA under **Security** in your Builder ID profile. The root MFA check in the course notebook measures a separate setting. [AWS Builder ID MFA](https://docs.aws.amazon.com/signin/latest/userguide/mfa-builder-id.html)

For a Paid project, review **Billing** → **Cost by project** → **Spend limit** in AWS Settings. If you set up budget alerts for the course, review those separately in **Billing and Cost Management** → **Budgets**. [AWS spend limits](https://docs.aws.amazon.com/accounts/latest/reference/create-spend-limit.html) · [AWS billing with the newer experience](https://docs.aws.amazon.com/accounts/latest/reference/supported-services-sign-up-new.html)

This cell reads the plan reported for the current AWS identity. If the lookup is denied, check the plan in AWS Settings and verify the terminal is using the intended project account.

In [ ]:
freetier = session.client("freetier")

try:
    plan = freetier.get_account_plan_state()
except freetier.exceptions.ResourceNotFoundException:
    print("No plan returned. Check your account in AWS Settings.")
else:
    print(f"Plan: {plan['accountPlanType']} ({plan['accountPlanStatus']})")

## Check AgentCore access in your project Region

Listing runtimes is a read-only check. An empty list is a successful result for a new project. [AgentCore regional endpoints](https://docs.aws.amazon.com/general/latest/gr/bedrock_agentcore.html)

In [ ]:
agentcore = session.client("bedrock-agentcore-control")
runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]

print("AgentCore runtime listing succeeded.")
if not runtimes:
    print("No runtimes were listed.")

## Before Day 2

The course deployment instructions target `us-east-1`. A successful runtime listing in Sydney confirms access to that operation in Sydney. Before deploying, check the course Region and the permissions for the resources you will create. AWS documents how advanced features can change the Region options for existing projects. [Activate advanced AWS features](https://docs.aws.amazon.com/accounts/latest/reference/activate-advanced-features.html)